# Genesis + Unitree Go2 + PPO — Colab GPU

目的：報酬から四脚の前進を学ぶ。**この構成でColab T4/Python3.13の導入・学習・評価・録画を確認済み**です。
評価のreset・移動距離計測を修正済み。既存結果は保存CSVから監査・訂正しました。
再生成した本ファイルの全セル再実行と、学習途中からの再開は未検証です。
Colabへこのファイルをアップロードし、**ランタイム → ランタイムのタイプを変更 → GPU**を選択してください。
T4を入口に、まず8環境のsmoke、次に1024環境で学習します。GPU割当・所要時間・歩行成功は保証しません。
速度は0.5m/sの前進一条件。旋回・外乱対応・実機への転送は含みません。

公式例のPPOと物理環境を使います。Genesis 1.4.5 / examples commit
`1ac23c39b32e567cc76e20a467fee99aaad257b7` / rsl-rl-lib 5.5.2。
古い記事のrsl_rl v1.0.2は混ぜません。PyTorchは2.11.0のCUDA12.8 wheelを使います。
直接依存は固定、推移依存は初回解決してfreezeを保存します。全依存lock済みという意味ではありません。
Linux Python3.10〜3.13向け。ColabのPythonが範囲外なら停止します。

公式資料：[歩行学習](https://genesis-world.readthedocs.io/en/v1.4.0/user_guide/policy_training/examples/locomotion.html)、
[導入](https://genesis-world.readthedocs.io/en/v1.4.0/user_guide/overview/installation.html)、
[固定ソース](https://github.com/Genesis-Embodied-AI/Genesis/tree/1ac23c39b32e567cc76e20a467fee99aaad257b7/examples/locomotion)。
GenesisはApache-2.0、rsl_rlはBSD-3-Clause。ロボット資産は配布元の個別条件も参照してください。
講義資料・認証鍵・private repoは不要です。取得先は公式GitHub / PyPI / PyTorchです。


## 1. 保存先・学習量

`USE_DRIVE=True`ならGoogle Driveの認証画面が出ます。専用フォルダにcheckpointと設定・ログを保存します。
FalseならColab切断で消えるため、最後にZIPをダウンロードしてください。
新しいセッションで再開する場合は同じセルから準備し直し、`RESUME_CHECKPOINT`へDrive上の**自分で作った**モデルの絶対パスを指定します。
モデルと同じフォルダのconfig.jsonが必要です。学習状態は復元しますが、乱数・物理状態を完全復元する再開ではありません。


In [ ]:
from pathlib import Path
import sys, os, json, subprocess, datetime, shutil, uuid, hashlib, urllib.request, difflib
assert (3, 10) <= sys.version_info[:2] < (3, 14), sys.version
assert sys.platform == "linux", "Use a Colab Linux GPU runtime"
BASE = Path("/content/genesis-go2")
BASE.mkdir(parents=True, exist_ok=True)
USE_DRIVE = True
NUM_ENVS = 1024       # OOMなら512、256へ。公式既定4096より小さく開始。
ITERATIONS = 301      # smokeとは別の本学習。学習不足ならcheckpointから追加。
SEED = 1
RESUME_CHECKPOINT = ""  # 例: /content/drive/MyDrive/genesis-go2/runs/.../model_100.pt
RECORD_VIDEO = True
BACKUP = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BACKUP = Path("/content/drive/MyDrive/genesis-go2/runs")
    BACKUP.mkdir(parents=True, exist_ok=True)
info = subprocess.check_output([
    "nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], text=True)
print(info)
(BASE / "gpu.txt").write_text(info)
(BASE / "python.txt").write_text(sys.version)
print("Local:", BASE, "Backup:", BACKUP)


## 2. 独立したPython環境

Notebookの既存Torchは置換せず、専用venvを作ります。数GBのダウンロードと初回compileが発生します。
GPUメモリ表示だけでCUDA12.8互換は確定しないため、最後に実際のCUDA tensor演算を確認します。
ここで失敗したら下へ進まず、setup.logとgpu.txtを確認してください。
OSパッケージは描画用共有ライブラリのみで、ドライバやEGL設定ファイルの書換えは行いません。


In [ ]:
PY = BASE / "venv/bin/python"
setup_log = BASE / "setup.log"
def setup(command):
    print(" ".join(map(str, command)), flush=True)
    with setup_log.open("a") as log:
        p = subprocess.run(list(map(str, command)), stdout=log, stderr=subprocess.STDOUT)
    if p.returncode:
        print(setup_log.read_text()[-12000:])
        raise RuntimeError(f"Setup exit={p.returncode}; see {setup_log}")
if not PY.exists():
    # Colab's Python may omit ensurepip; uv seeds the venv without changing the notebook kernel.
    setup([sys.executable, "-m", "pip", "install", "--target", BASE / "bootstrap", "uv==0.9.2"])
    setup([BASE / "bootstrap/bin/uv", "venv", "--python", sys.executable,
           "--seed", "--cache-dir", BASE / "uv-cache", BASE / "venv"])
setup(["apt-get", "update", "-qq"])
setup(["apt-get", "install", "-y", "-qq", "libegl1", "libgl1", "libglib2.0-0"])
setup([PY, "-m", "pip", "install", "pip==25.3"])
setup([PY, "-m", "pip", "install", "torch==2.11.0+cu128",
       "--index-url", "https://download.pytorch.org/whl/cu128",
       "--report", str(BASE / "torch-install.json")])


In [ ]:
(BASE / "requirements.txt").write_text('genesis-world==1.4.5\nrsl-rl-lib==5.5.2\ntensorboard==2.20.0\n# torch==2.11.0+cu128 is installed separately from the official PyTorch index.\n# Transitive dependencies are resolved on Colab and recorded with pip freeze.\n')

setup([PY, "-m", "pip", "install", "-r", BASE / "requirements.txt",
       "--report", str(BASE / "genesis-install.json")])
setup([PY, "-m", "pip", "check"])
(BASE / "requirements.freeze.txt").write_text(subprocess.check_output(
    [str(PY), "-m", "pip", "freeze"], text=True))
setup([PY, "-c", "import torch; assert torch.cuda.is_available(); "
       "print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name()); "
       "print((torch.ones(8, device='cuda') * 2).sum().item())"])
print("Setup OK. Freeze and pip download hashes recorded.")


## 3. 公式例の取得・録画用の最小差分

固定commitから2ファイルを取得しSHA-256を検証します。原ファイルは保持し、別コピーにcamera追加だけを適用します。
差分はcamera.patchへ保存します。reward・観測・制御は変更しません。
初期位置は[0,0,0.42]m、quaternionはwxyz=[1,0,0,0]。12関節はFR/FL/RR/RLのhip→thigh→calf順です。
角度rad、関節速度rad/s、制御50Hz、物理substep2、actionは基準角への0.25rad倍率の残差です。


In [ ]:
SPEC = {'repository': 'https://github.com/Genesis-Embodied-AI/Genesis', 'commit': '1ac23c39b32e567cc76e20a467fee99aaad257b7', 'engine_version': '1.4.5', 'files': {'go2_env.py': '22a08b9f1f5c765732a7b3d04c98b3cb6a25b565aa024479a0526bbc654c1a63', 'go2_train.py': '9d78b33a90e8833be648c302a818f1b10ebfb632d0f5de5214f68bf1d714cb00'}}

SOURCE = BASE / "source"
SOURCE.mkdir(exist_ok=True)
for name, expected in SPEC["files"].items():
    url = f'https://raw.githubusercontent.com/Genesis-Embodied-AI/Genesis/{SPEC["commit"]}/examples/locomotion/{name}'
    if (SOURCE / name).exists():
        data = (SOURCE / name).read_bytes()
    else:
        with urllib.request.urlopen(url, timeout=60) as response:
            data = response.read()
    assert hashlib.sha256(data).hexdigest() == expected, f"Source mismatch: {name}"
    (SOURCE / name).write_bytes(data)
(SOURCE / "upstream.lock.json").write_text(json.dumps(SPEC, indent=2))
original = (SOURCE / "go2_env.py").read_text()
anchor = "        self.scene.build(n_envs=num_envs)"
signature = "show_viewer=False):"
assert original.count(anchor) == original.count(signature) == 1
modified = original.replace(signature, "show_viewer=False, record=False):").replace(anchor,
    "        self.camera = None\n"
    "        if record:\n"
    "            self.camera = self.scene.add_camera(res=(640, 480), pos=(2, -2, 1.5), "
    "lookat=(0, 0, 0.3), GUI=False, env_idx=0)\n" + anchor)
(SOURCE / "go2_record_env.py").write_text(modified)
(SOURCE / "camera.patch").write_text("".join(difflib.unified_diff(
    original.splitlines(True), modified.splitlines(True), fromfile="go2_env.py", tofile="go2_record_env.py")))
print((SOURCE / "camera.patch").read_text())


In [ ]:
(SOURCE / "worker.py").write_text('"""Bounded CUDA training/evaluation; embedded in the standalone Colab notebook."""\nimport argparse\nimport copy\nimport csv\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport random\nimport shutil\nimport time\n\n\ndef write_json(path, value):\n    path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\\n")\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("mode", choices=["smoke", "train", "eval"])\n    parser.add_argument("--run", type=Path, required=True)\n    parser.add_argument("--backup", type=Path)\n    parser.add_argument("--checkpoint", type=Path)\n    parser.add_argument("--num-envs", type=int, default=1024)\n    parser.add_argument("--iterations", type=int, default=301)\n    parser.add_argument("--seed", type=int, default=1)\n    parser.add_argument("--video", action="store_true")\n    args = parser.parse_args()\n    if args.num_envs < 1 or args.iterations < 1:\n        parser.error("num-envs and iterations must be positive")\n    # Never clear or reuse a run directory.\n    args.run.mkdir(parents=True, exist_ok=False)\n    # Colab exports an inline backend unavailable inside the isolated venv.\n    # Set this before Genesis imports matplotlib transitively.\n    os.environ["MPLBACKEND"] = "Agg"\n    import numpy as np\n    import torch\n    import genesis as gs\n    from go2_record_env import Go2Env\n    from go2_train import get_cfgs, get_train_cfg\n    from rsl_rl.runners import OnPolicyRunner\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA GPU required. Select a Colab GPU runtime.")\n    torch.set_num_threads(2)\n    gs.init(backend=gs.cuda, precision="32", logging_level="warning",\n            seed=args.seed, performance_mode=True)\n    if args.checkpoint:\n        args.checkpoint = args.checkpoint.resolve()\n        saved = json.loads((args.checkpoint.parent / "config.json").read_text())\n        current = json.loads((Path(__file__).parent / "upstream.lock.json").read_text())\n        if saved["upstream"] != current:\n            raise RuntimeError("Checkpoint and current upstream pins differ")\n        cfgs = saved["cfgs"]\n        train_cfg = saved["train_cfg"]\n    else:\n        cfgs = get_cfgs()\n        train_cfg = get_train_cfg(args.run.name)\n        train_cfg["save_interval"] = 25\n    spec = json.loads((Path(__file__).parent / "upstream.lock.json").read_text())\n    config = {"args": {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()},\n              "cfgs": cfgs, "train_cfg": train_cfg, "upstream": spec,\n              "control_dt_s": 0.02, "physics_substeps": 2,\n              "success_rule": "500 steps without reset, vx MAE <0.2 m/s, x displacement >1m",\n              "evaluation_seeds": [1000, 1001, 1002]}\n    write_json(args.run / "config.json", config)\n    write_json(args.run / "dependencies.json", {\n        p: importlib.metadata.version(p) for p in\n        ["genesis-world", "rsl-rl-lib", "torch", "quadrants", "tensorboard"]})\n    for name in ["worker.py", "go2_env.py", "go2_record_env.py", "go2_train.py", "upstream.lock.json"]:\n        shutil.copy2(Path(__file__).parent / name, args.run / name)\n\n    def backup():\n        if args.backup:\n            destination = args.backup / args.run.name\n            destination.mkdir(parents=True, exist_ok=True)\n            for path in args.run.iterdir():\n                if path.is_file():\n                    temp = destination / (path.name + ".partial")\n                    shutil.copy2(path, temp)\n                    temp.replace(destination / path.name)\n\n    class SavedRunner(OnPolicyRunner):\n        def save(self, path, infos=None):\n            super().save(path, infos)\n            backup()\n\n    n = 8 if args.mode == "smoke" else (1 if args.mode == "eval" else args.num_envs)\n    env = Go2Env(n, *copy.deepcopy(cfgs), record=args.video)\n    obs = env.reset()\n    assert obs["policy"].shape == (n, 45)\n    runner = SavedRunner(env, copy.deepcopy(train_cfg), str(args.run), device=gs.device)\n    if args.checkpoint:\n        runner.load(str(args.checkpoint), map_location=str(gs.device))\n        # Upstream stores the index of the last completed iteration.\n        runner.current_learning_iteration += 1\n    started = time.monotonic()\n    if args.mode in ["smoke", "train"]:\n        policy = runner.alg.get_policy()\n        before = [p.detach().clone() for p in policy.parameters()]\n        iterations = 2 if args.mode == "smoke" else args.iterations\n        runner.learn(num_learning_iterations=iterations, init_at_random_ep_len=True)\n        delta = sum((a - b).abs().sum().item() for a, b in zip(policy.parameters(), before))\n        assert delta > 0 and np.isfinite(delta), "PPO did not update finite parameters"\n        runner.save(str(args.run / "checkpoint.pt"))\n        metrics = {"status": "ppo_updates_completed", "iterations_this_run": iterations,\n                   "transitions_this_run": n * train_cfg["num_steps_per_env"] * iterations,\n                   "parameter_l1_change": delta, "walking_success": "not_evaluated"}\n    else:\n        if not args.checkpoint:\n            raise ValueError("eval requires --checkpoint")\n        policy = runner.get_inference_policy(device=gs.device)\n        rows, trials = [], []\n        # Identical initial pose and command; seed changes alone do not establish generalization.\n        for control in ["standing_pd", "ppo"]:\n            for trial_seed in [1000, 1001, 1002]:\n                random.seed(trial_seed)\n                np.random.seed(trial_seed)\n                torch.manual_seed(trial_seed)\n                obs = env.reset()\n                # Upstream reset uses skip_forward=True. get_pos() can still expose\n                # the previous trial\'s cached pose until the first physics step.\n                origin = np.asarray(env.env_cfg["base_init_pos"], dtype=float).copy()\n                errors = []\n                fell = False\n                writer = None\n                # Streaming avoids keeping 250 RGB frames in RAM.\n                if args.video:\n                    import av\n                    video = args.run / f"{control}-{trial_seed}.mp4"\n                    writer = av.open(str(video), "w")\n                    stream = writer.add_stream("libx264", rate=25)\n                    stream.width, stream.height, stream.pix_fmt = 640, 480, "yuv420p"\n                try:\n                    # Environment buffers survive across trials; inference tensors cannot\n                    # be mutated by the next reset outside InferenceMode.\n                    with torch.no_grad():\n                        for step in range(500):\n                            actions = policy(obs) if control == "ppo" else torch.zeros((1, 12), device=gs.device)\n                            obs, rewards, dones, _ = env.step(actions)\n                            if not torch.isfinite(obs["policy"]).all() or not torch.isfinite(rewards).all():\n                                raise RuntimeError("Non-finite rollout")\n                            done = bool(dones[0].item())\n                            velocity = env.base_lin_vel[0].detach().cpu().tolist()\n                            pos = env.robot.get_pos()[0].detach().cpu().numpy()\n                            rows.append([control, trial_seed, step, (step + 1) * 0.02,\n                                         *velocity, *pos.tolist(), float(rewards[0]), done])\n                            # Exclude initial 1 second of settling; never count a reset state as tracking.\n                            if step >= 50 and not done:\n                                errors.append(abs(velocity[0] - 0.5))\n                            if writer and step % 2 == 0:\n                                env.camera.set_pose(pos=pos + np.array([1.7, -2.0, 1.2]),\n                                                    lookat=pos + np.array([0, 0, -0.1]))\n                                rgb = env.camera.render()[0]\n                                for packet in stream.encode(av.VideoFrame.from_ndarray(rgb, format="rgb24")):\n                                    writer.mux(packet)\n                            if done:\n                                fell = True\n                                break\n                finally:\n                    if writer:\n                        for packet in stream.encode():\n                            writer.mux(packet)\n                        writer.close()\n                mae = float(np.mean(errors)) if errors else None\n                displacement = float(pos[0] - origin[0]) if not fell else None\n                trials.append({"control": control, "seed": trial_seed, "steps": step + 1,\n                               "terminated_early": fell, "vx_mae_m_s": mae,\n                               "x_displacement_m": displacement,\n                               "success": not fell and mae is not None and mae < 0.2 and displacement > 1.0})\n        with (args.run / "rollout.csv").open("w") as output:\n            csvwriter = csv.writer(output)\n            csvwriter.writerow(["control", "seed", "step", "time_s", "vx", "vy", "vz",\n                                "x", "y", "z", "reward", "done"])\n            csvwriter.writerows(rows)\n        metrics = {"status": "evaluated", "trials": trials,\n                   "ppo_success_rate": sum(t["success"] for t in trials if t["control"] == "ppo") / 3,\n                   "scope": "fixed initial pose, fixed forward command; no turning or disturbance test"}\n    metrics["elapsed_s"] = time.monotonic() - started\n    write_json(args.run / "metrics.json", metrics)\n    backup()\n    print(json.dumps(metrics, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n')
print("Worker prepared")


## 4. 実行・ログ保存関数

各処理は新しいプロセス・runディレクトリで実行し、失敗時も終了コードとログを残します。
checkpointは25更新ごと・終了時に保存し、Drive有効時は専用フォルダへコピーします。
中断や突然の切断では最後にコピーが完了したcheckpointから再開します。


In [ ]:
def launch(mode, *extra, timeout=3600):
    run = BASE / "runs" / (datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
                          + "-" + mode + "-" + uuid.uuid4().hex[:6])
    # Worker creates the run itself. Parent logs live in a separate sibling folder.
    logdir = BASE / "launches" / run.name
    logdir.mkdir(parents=True)
    command = [str(PY), "-u", str(SOURCE / "worker.py"), mode,
               "--run", str(run), "--seed", str(SEED), *map(str, extra)]
    if BACKUP:
        command += ["--backup", str(BACKUP)]
    (logdir / "command.json").write_text(json.dumps(command, indent=2))
    env = dict(os.environ, MPLBACKEND="Agg", PYOPENGL_PLATFORM="egl", PYTHONDONTWRITEBYTECODE="1",
               OMP_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2")
    code = -1
    try:
        # Log to disk, poll progress without accumulating output in notebook memory.
        import time
        with (logdir / "run.log").open("w") as log:
            process = subprocess.Popen(command, cwd=SOURCE, env=env, stdout=log, stderr=subprocess.STDOUT)
            start = time.monotonic()
            offset = 0
            try:
                while process.poll() is None:
                    time.sleep(5)
                    with (logdir / "run.log").open() as reader:
                        reader.seek(offset)
                        tail = reader.read()
                        offset = reader.tell()
                    if tail:
                        print(tail[-5000:], end="", flush=True)
                    if time.monotonic() - start > timeout:
                        raise TimeoutError(f"{mode}: exceeded {timeout}s")
                code = process.returncode
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                code = process.returncode
                raise
    finally:
        (logdir / "exit-code.txt").write_text(str(code))
        run.mkdir(parents=True, exist_ok=True)
        for p in logdir.iterdir():
            shutil.copy2(p, run / p.name)
        for name in ["gpu.txt", "python.txt", "requirements.freeze.txt", "requirements.txt",
                     "torch-install.json", "genesis-install.json", "setup.log"]:
            if (BASE / name).exists():
                shutil.copy2(BASE / name, run / name)
        shutil.copy2(SOURCE / "camera.patch", run / "camera.patch")
        if BACKUP:
            shutil.copytree(run, BACKUP / run.name, dirs_exist_ok=True)
    print((logdir / "run.log").read_text()[-2500:])
    if code:
        raise RuntimeError(f"{mode} exit={code}. Inspect {run / 'run.log'}")
    print("Saved:", run)
    return run


## 5. Smoke → 本学習

smokeは8環境×24step×2更新＝384 transition。有限値とPPOの重み更新を確認するだけで、歩行成功とは扱いません。
本学習の初期値は1024×24×301＝約740万transitionです。1時間で停止する上限を設けています。
初回compileに時間がかかります。OOMならNUM_ENVSを下げ、新しいrunで再試行してください。


In [ ]:
SMOKE_RUN = launch("smoke", timeout=1200)
smoke = json.loads((SMOKE_RUN / "metrics.json").read_text())
assert smoke["status"] == "ppo_updates_completed"
SMOKE_OK = True


In [ ]:
assert globals().get("SMOKE_OK"), "Run smoke successfully first"
resume = ["--checkpoint", RESUME_CHECKPOINT] if RESUME_CHECKPOINT else []
TRAIN_RUN = launch("train", "--num-envs", NUM_ENVS, "--iterations", ITERATIONS,
                   *resume, timeout=3600)
CHECKPOINT = TRAIN_RUN / "checkpoint.pt"
print("Evaluation / resume checkpoint:", CHECKPOINT)


## 6. 固定条件で評価・録画

各10秒・3seedで立位PDとPPOを比較します。同じ初期姿勢・前進0.5m/sであり、3seedは多様な初期条件の検証ではありません。
成功は「500stepでresetなし、最初1秒を除く前進速度MAE<0.2m/s、前方へ1m以上」です。
転倒（roll/pitchが10度超）・数値異常等で終了し、resetを跨いで成功を水増ししません。
CSVの速度は体座標m/s、位置は世界座標m。MP4は640×480・25fps・実時間1倍です。
EGL録画だけ失敗したらRECORD_VIDEO=Falseでこのセルを再実行し、まず数値評価を残してください。
これらは短い確認動画で、最終提出の1〜3分動画は別途編集が必要です。


In [ ]:
# 評価のみ再開する場合は、Drive上の自作checkpointへ変更できます。
# CHECKPOINT = Path("/content/drive/MyDrive/genesis-go2/runs/.../checkpoint.pt")
EVAL_RUN = launch("eval", "--checkpoint", CHECKPOINT,
                  *(["--video"] if RECORD_VIDEO else []), timeout=1800)
print((EVAL_RUN / "metrics.json").read_text())
from IPython.display import display, Video
for video in sorted(EVAL_RUN.glob("*.mp4")):
    print(video.name)
    display(Video(str(video), embed=True))


## 7. 保存・ダウンロード

Drive有効時も、必要ならZIPを手元へ保存します。checkpoint/config/依存記録/評価CSV/動画/ログをまとめます。
重み・動画・実行済みNotebookを講義リポジトリへコミットしないでください。
続けるときは次の優先順です：歩行の評価 → 学習量調整 → 速度の範囲変更 → 旋回。
command分布を変えるときは新規学習として設定と評価条件を作り直します。


In [ ]:
archive = shutil.make_archive(str(BASE / "genesis-go2-results"), "zip", BASE / "runs")
print(archive)
from google.colab import files
files.download(archive)
